# BrandSphere AI — Week 5: Color Palette & Visual Harmony Engine
**CRS AI Capstone 2025-26 | Scenario 1**

This notebook:
1. Loads logo images from the dataset
2. Uses KMeans clustering to extract dominant colors
3. Maps colors to industry and brand personality
4. Saves palette recommendations as CSV

## Cell 1 — Install Dependencies

In [ ]:
!pip install kagglehub opencv-python-headless scikit-learn matplotlib seaborn numpy pandas pillow -q
print('✅ Dependencies installed')

## Cell 2 — Load Logo Images from Dataset

In [ ]:
import kagglehub, os, cv2, random
import numpy as np
import matplotlib.pyplot as plt

print('📥 Loading logo dataset...')
logo_path = kagglehub.dataset_download('siddharthkumarsah/logo-dataset-2341-classes-and-167140-images')
print(f'✅ Dataset path: {logo_path}')

# Find dataset root
dataset_root = None
for root, dirs, files in os.walk(logo_path):
    if len(dirs) > 20:
        dataset_root = root
        break
if dataset_root is None:
    dataset_root = logo_path

classes = sorted([d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))])
print(f'📊 Classes available: {len(classes)}')

## Cell 3 — KMeans Color Extraction Function

In [ ]:
from sklearn.cluster import KMeans
from PIL import Image
import io

def extract_dominant_colors(img_array: np.ndarray, n_colors: int = 5) -> list:
    """
    Use KMeans clustering to extract dominant colors from a logo image.
    Returns list of HEX color codes.
    """
    # Reshape image to list of pixels
    img_rgb = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
    pixels = img_rgb.reshape(-1, 3).astype(np.float32)

    # Remove near-white and near-black pixels (background)
    mask = ~(((pixels > 230).all(axis=1)) | ((pixels < 25).all(axis=1)))
    filtered = pixels[mask]

    if len(filtered) < n_colors * 10:
        filtered = pixels  # fallback if too many filtered

    # KMeans clustering
    kmeans = KMeans(n_clusters=n_colors, random_state=42, n_init=10)
    kmeans.fit(filtered)

    # Sort by cluster size (most dominant first)
    counts = np.bincount(kmeans.labels_)
    sorted_idx = np.argsort(counts)[::-1]
    colors = kmeans.cluster_centers_[sorted_idx]

    # Convert to HEX
    hex_colors = ['#{:02x}{:02x}{:02x}'.format(int(r), int(g), int(b))
                  for r, g, b in colors]
    return hex_colors

print('✅ KMeans color extractor defined')

## Cell 4 — Extract Colors from Sample Logos

In [ ]:
# Extract palettes from 30 random logos
SAMPLE_COUNT = 30
all_palettes = {}

sampled_classes = random.sample(classes, min(SAMPLE_COUNT, len(classes)))

for cls in sampled_classes:
    cls_dir = os.path.join(dataset_root, cls)
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if not imgs:
        continue
    img_path = os.path.join(cls_dir, random.choice(imgs))
    img = cv2.imread(img_path)
    if img is None:
        continue
    img = cv2.resize(img, (128, 128))
    try:
        palette = extract_dominant_colors(img, n_colors=5)
        all_palettes[cls] = palette
        print(f'✅ {cls}: {palette}')
    except Exception as e:
        print(f'⚠️  {cls}: {e}')

print(f'\n✅ Extracted palettes for {len(all_palettes)} logos')

## Cell 5 — Visualize Color Swatches

In [ ]:
def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16)/255 for i in (0, 2, 4))

# Show palettes for first 10 logos
sample_items = list(all_palettes.items())[:10]
fig, axes = plt.subplots(len(sample_items), 6, figsize=(14, 2.2*len(sample_items)))
fig.suptitle('KMeans Extracted Color Palettes from Logo Dataset', fontsize=14, fontweight='bold')

for row, (cls, palette) in enumerate(sample_items):
    # Load logo thumbnail
    cls_dir = os.path.join(dataset_root, cls)
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    img = cv2.imread(os.path.join(cls_dir, imgs[0]))
    img_rgb = cv2.cvtColor(cv2.resize(img, (64,64)), cv2.COLOR_BGR2RGB)
    axes[row][0].imshow(img_rgb)
    axes[row][0].set_title(cls[:10], fontsize=8)
    axes[row][0].axis('off')

    for col, color in enumerate(palette[:5]):
        axes[row][col+1].set_facecolor(color)
        axes[row][col+1].set_title(color, fontsize=7)
        axes[row][col+1].axis('off')

plt.tight_layout()
plt.savefig('color_palettes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Color palette visualization saved')

## Cell 6 — Color-Personality & Industry Mapping

In [ ]:
import pandas as pd

# Industry color psychology mapping
INDUSTRY_PALETTE = {
    'Tech / Software':     {'colors': ['#1B3A6B','#2E86AB','#E8F4FD','#F4A261','#2D6A4F'], 'rationale': 'Blue = trust/innovation, Orange = energy'},
    'Fashion / Apparel':   {'colors': ['#1A1A2E','#C9A84C','#E8DCC8','#8B4513','#F5F0E8'], 'rationale': 'Dark + gold = luxury, beige = warmth'},
    'Food & Beverage':     {'colors': ['#E63946','#F4A261','#2D6A4F','#FFFFFF','#1A1A2E'], 'rationale': 'Red = appetite, orange = warmth, green = fresh'},
    'Healthcare':          {'colors': ['#0077B6','#00B4D8','#90E0EF','#CAF0F8','#FFFFFF'], 'rationale': 'Blues = trust, calm, cleanliness'},
    'Finance':             {'colors': ['#1B3A6B','#2E4057','#C9A84C','#F0F4F8','#FFFFFF'], 'rationale': 'Navy = authority, gold = prosperity'},
    'Education':           {'colors': ['#4361EE','#7209B7','#F72585','#4CC9F0','#FFFFFF'], 'rationale': 'Vibrant = creativity and curiosity'},
    'Retail / E-commerce': {'colors': ['#E63946','#F4A261','#1A1A2E','#FFFFFF','#2D6A4F'], 'rationale': 'Red = urgency/sale, orange = impulse buy'},
    'Real Estate':         {'colors': ['#1B3A6B','#C9A84C','#F0EDE8','#2D6A4F','#1A1A2E'], 'rationale': 'Blue = trust, gold = premium value'},
    'Creative / Design':   {'colors': ['#F72585','#7209B7','#4361EE','#4CC9F0','#FFFFFF'], 'rationale': 'Bold saturated = creativity, expression'},
    'Manufacturing':       {'colors': ['#1A1A2E','#2E4057','#C9A84C','#7A7A85','#F0F4F8'], 'rationale': 'Dark = strength, grey = precision'},
}

# Personality color modifiers
PERSONALITY_MODIFIER = {
    'minimalist': 'Desaturate by 40%, increase white space',
    'vibrant':    'Increase saturation by 30%, add complementary accent',
    'luxury':     'Replace palette with: #1A1A1A, #C9A84C, #E8DCC8, #8B6914, #F5F0E8',
    'bold':       'Increase contrast, darken primary by 20%',
    'elegant':    'Soften saturation by 20%, add muted tones',
}

# Build recommendation dataframe
rows = []
for industry, data in INDUSTRY_PALETTE.items():
    rows.append({
        'Industry': industry,
        'Primary':    data['colors'][0],
        'Secondary':  data['colors'][1],
        'Accent':     data['colors'][2],
        'Background': data['colors'][3],
        'Text':       data['colors'][4],
        'Psychology': data['rationale'],
    })

df_palette = pd.DataFrame(rows)
df_palette.to_csv('color_palette_recommendations.csv', index=False)
print('📋 Color Palette Recommendations:')
print(df_palette[['Industry','Primary','Secondary','Accent','Psychology']].to_string(index=False))
print('\n✅ Saved to color_palette_recommendations.csv')

## Cell 7 — Distribution Plot: Color Categories

In [ ]:
import seaborn as sns

def classify_hue(hex_color):
    """Classify a hex color into a named hue category."""
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    mx = max(r, g, b)
    if mx < 50:    return 'Black/Dark'
    if mx > 220 and min(r,g,b) > 200: return 'White/Light'
    if r > g and r > b:   return 'Red/Orange'
    if g > r and g > b:   return 'Green'
    if b > r and b > g:   return 'Blue'
    if r > 150 and g > 150 and b < 80: return 'Yellow/Gold'
    return 'Neutral/Gray'

# Count hue categories across extracted palettes
hue_counts = {}
for cls, palette in all_palettes.items():
    for color in palette:
        hue = classify_hue(color)
        hue_counts[hue] = hue_counts.get(hue, 0) + 1

plt.figure(figsize=(10, 5))
bars = plt.bar(hue_counts.keys(), hue_counts.values(),
               color=['#1A1A2E','#E8F0F5','#E63946','#2D6A4F','#2E86AB','#C9A84C','#7A7A85'])
plt.title('Color Category Distribution in Logo Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Color Category')
plt.ylabel('Frequency')
plt.xticks(rotation=15)
for bar, val in zip(bars, hue_counts.values()):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(val), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('color_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Color distribution chart saved')

## Cell 8 — Save & Upload to Drive

In [ ]:
import pickle, shutil, json
from google.colab import drive

# Save palette data
with open('palette_data.pkl', 'wb') as f:
    pickle.dump({'extracted_palettes': all_palettes,
                 'industry_palette': INDUSTRY_PALETTE,
                 'personality_modifier': PERSONALITY_MODIFIER}, f)

drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/BrandSphere_Models'
os.makedirs(save_dir, exist_ok=True)

for fname in ['palette_data.pkl', 'color_palette_recommendations.csv',
              'color_palettes.png', 'color_distribution.png']:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(save_dir, fname))
        print(f'   ✅ Uploaded {fname}')

## Cell 9 — Summary

In [ ]:
print('=' * 55)
print('  BRANDSPHERE AI — COLOR PALETTE ENGINE SUMMARY')
print('=' * 55)
print(f'  Dataset:       Logo Dataset (Kaggle)')
print(f'  Logos sampled: {len(all_palettes)}')
print(f'  Method:        KMeans clustering (k=5)')
print(f'  Colors/logo:   5 dominant colors extracted')
print(f'  Preprocessing: Remove near-white/black background')
print(f'  Integration:   Industry + personality → palette')
print('=' * 55)
print('  OUTPUTS:')
print('  - palette_data.pkl                    → pickled data')
print('  - color_palette_recommendations.csv  → lookup table')
print('  - color_palettes.png                 → swatch viz')
print('  - color_distribution.png             → hue stats')
print('=' * 55)